# Customer Intelligence System using Clustering

## Objective
Segment customers based on services, tenure, and charges to identify key customer groups using K-Means, DBSCAN and PCA visualization.

# 1. Environment Setup
Required libraries: pandas, numpy, matplotlib, seaborn, scikit-learn

# 2. Import Required Libraries and Load Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 3. Dataset Overview

In [3]:
df.shape

(7043, 21)

In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Dataset contains 7043 customers and 21 attributes.
The features represent customer demographics, services subscribed, and charges.
Some missing values in TotalCharges will be cleaned during preprocessing.
The dataset is suitable for customer segmentation using clustering.

# 4. Data Cleaning and Preprocessing

In [8]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [9]:
# Convert TotalCharges to numeric and drop rows with missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Save customerID
customer_ids = df['customerID']

# Exclude customerID from features
X = df.drop('customerID', axis=1)

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# 5. Feature Scaling using StandardScaler

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_encoded)

In [11]:
print(X_scaled.shape)

(7032, 31)


Data preprocessing was performed before clustering.
The customerID column was excluded as it is a unique identifier.
Categorical variables were one-hot encoded.
StandardScaler was applied to normalize all features so that each feature contributes equally during clustering.

# 6. Correlation Analysis

In [12]:
plt.figure(figsize=(12,10))
sns.heatmap(X_encoded.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# 7. Finding Optimal Clusters using Elbow Method

In [13]:
from sklearn.cluster import KMeans

In [14]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    wcss.append(kmeans.inertia_)

In [15]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(range(1,11), wcss, marker='o')

plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')

plt.grid(True)

plt.show()

The Elbow Method was used to determine the optimal number of clusters.
The graph shows a bend around k = 3.
Therefore, 3 clusters were selected for customer segmentation.

# 8. K-Means Clustering

In [16]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

In [17]:
df['kmeans_cluster'].value_counts()

kmeans_cluster
0    3136
2    2376
1    1520
Name: count, dtype: int64

# 9. Silhouette Score Evaluation

In [18]:
from sklearn.metrics import silhouette_score

score = silhouette_score(
    X_scaled,
    df['kmeans_cluster']
)

print("Silhouette Score:", score)

Silhouette Score: 0.2169046816963205


The Silhouette Score obtained is approximately 0.2169, indicating reasonable cluster separation and supporting the selected cluster configuration (k = 3).

# 10. Density-Based Clustering using DBSCAN

In [19]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=3.0,
    min_samples=10
)

df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

In [20]:
df['dbscan_cluster'].value_counts()

dbscan_cluster
 0    2986
-1    2296
 2    1514
 1     203
 3      33
Name: count, dtype: int64

DBSCAN clustering was applied to identify density-based clusters.
Unlike K-Means, DBSCAN can detect noise points and clusters of arbitrary shape.
The results provide an alternative clustering perspective for customer segmentation.

# 11. PCA-Based Cluster Visualization

In [21]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

In [22]:
pca_df = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2']
)

pca_df['Cluster'] = df['kmeans_cluster']

In [23]:
plt.figure(figsize=(10,8))
sns.scatterplot(
    data=pca_df,
    x='PC1',
    y='PC2',
    hue='Cluster',
    palette='viridis',
    alpha=0.6
)
plt.title('PCA-Based Cluster Visualization')
plt.show()

The PCA scatterplot provides a two-dimensional representation of the customer data and shows visually how the 3 clusters are separated.

# 12. Cluster Profiling

In [24]:
cluster_profile = df.groupby('kmeans_cluster').mean(numeric_only=True)

cluster_profile

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,dbscan_cluster
kmeans_cluster,,,,,
0,0.206633,16.064094,67.694292,1073.474506,-0.226403
1,0.034211,30.667763,21.076283,665.220329,1.988158
2,0.186027,55.133838,88.946023,4915.243161,-0.537879


# 13. Observations

### Observation 1
Cluster 0 represents relatively new/short-term customers (average tenure ~16 months) with moderate monthly charges (~$68) and a very high churn rate (46.4%). These represent high-risk customers requiring proactive retention strategies.

### Observation 2
Cluster 1 represents cost-sensitive but loyal customers with low monthly charges (~$21), moderate tenure (~31 months), and a very low churn rate (7.4%). These are typically basic service users who are highly stable.

### Observation 3
Cluster 2 represents high-value, long-term premium customers (average tenure ~55 months) with high monthly charges (~$89) and a low churn rate (12.7%). These are the most valuable customers with high lifetime value.

### Observation 4
The Silhouette Score of ~0.217 confirms that the customer segmentation has a reasonable cluster structure.

### Observation 5
DBSCAN successfully identified outliers (noise points) representing customers with atypical service combinations or charges.

# 14. Conclusion

This project successfully applied unsupervised learning techniques to segment telco customers. K-Means clustering identified three distinct customer segments: risky new customers, budget-loyal customers, and high-value premium customers. The Silhouette Score validated the cluster structure, DBSCAN detected unique outlier customers, and PCA visualization provided a clear representation of customer segmentation. The results can support marketing campaigns, churn prevention strategies, and data-driven business decisions.